# Submission 07: Heavy CatBoost Ensemble

This submission is based on Experiment 22, where the equal-weight CAT_A + CAT_B ensemble achieved the best OOF accuracy of 0.8406.

Both CatBoost models are retrained on the full training dataset before generating predictions for the Kaggle test set.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

BASE_DIR = Path('C:/Users/aakif/Documents/DataCompetition2')
DATA_DIR = BASE_DIR / 'data'
SUB_DIR = BASE_DIR / 'submissions'
SUB_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)

In [ ]:
def clean_title(title):
    title = str(title).strip()
    title = {'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'}.get(title, title)
    common = {'Mr', 'Miss', 'Mrs', 'Master'}
    return title if title in common else 'Rare'


def engineer_features(df, reference=None):
    out = df.copy()
    ref = out.copy() if reference is None else reference.copy()

    out['FamilySize'] = out['SibSp'] + out['Parch'] + 1
    out['IsAlone'] = (out['FamilySize'] == 1).astype(int)

    out['Title'] = out['Name'].str.extract(r',\s*([^.]*)\.', expand=False).fillna('Rare').map(clean_title)
    out['Surname'] = out['Name'].str.split(',').str[0].str.strip()

    out['CabinDeck'] = out['Cabin'].fillna('Unknown').astype(str).str[0]
    out['HasCabin'] = out['Cabin'].notna().astype(int)
    out['DeckKnown'] = out['HasCabin']
    out['CabinCount'] = out['Cabin'].fillna('').astype(str).str.count(' ') + out['HasCabin']

    out['TicketPrefix'] = (
        out['Ticket'].astype(str)
        .str.replace(r'\d', '', regex=True)
        .str.replace(r'[./]', '', regex=True)
        .str.replace(' ', '', regex=True)
        .replace('', 'NONE')
    )

    ref['Surname'] = ref['Name'].str.split(',').str[0].str.strip()
    out['TicketGroupSize'] = out['Ticket'].map(ref['Ticket'].value_counts()).fillna(1)
    out['SurnameGroupSize'] = out['Surname'].map(ref['Surname'].value_counts()).fillna(1)

    out['FarePerPerson'] = out['Fare'] / out['TicketGroupSize'].replace(0, 1)
    out['FarePerAge'] = out['Fare'] / out['Age'].clip(lower=1)
    out['ClassFare'] = out['Pclass'] * out['Fare']
    out['FamilyFare'] = out['Fare'] / out['FamilySize'].replace(0, 1)

    out['AgeMissing'] = out['Age'].isna().astype(int)
    out['FareMissing'] = out['Fare'].isna().astype(int)
    out['EmbarkedMissing'] = out['Embarked'].isna().astype(int)
    out['FarePerPersonMissing'] = out['FarePerPerson'].isna().astype(int)

    out['Child'] = ((out['Age'] < 16) & out['Age'].notna()).astype(int)
    out['Mother'] = ((out['Sex'] == 'female') & (out['Parch'] > 0) & (out['Age'] > 18)).astype(int)
    out['LargeFamily'] = (out['FamilySize'] >= 5).astype(int)
    out['SmallFamily'] = out['FamilySize'].between(2, 4).astype(int)
    out['FemaleChild'] = ((out['Sex'] == 'female') & (out['Age'] < 16)).astype(int)

    out['Sex_Pclass'] = out['Sex'].astype(str) + '_' + out['Pclass'].astype(str)
    out['FamilySex'] = out['Sex'].astype(str) + '_' + out['FamilySize'].astype(str)
    out['PclassTitle'] = out['Pclass'].astype(str) + '_' + out['Title'].astype(str)
    out['FamilyTicket'] = out['FamilySize'].astype(str) + '_' + out['TicketPrefix'].astype(str)
    out['SexTitle'] = out['Sex'].astype(str) + '_' + out['Title'].astype(str)

    out['SiblingChildRatio'] = out['SibSp'] / (out['Parch'] + 1)
    out['NameLength'] = out['Name'].astype(str).str.len()
    out['NameWords'] = out['Name'].astype(str).str.split().str.len()
    out['TicketLength'] = out['Ticket'].astype(str).str.len()

    out['FamilySizeBand'] = pd.cut(
        out['FamilySize'],
        bins=[0, 1, 4, 7, np.inf],
        labels=['Alone', 'Small', 'Medium', 'Large']
    ).astype(str)

    out['AgeBand'] = pd.cut(
        out['Age'],
        bins=[-np.inf, 5, 12, 18, 30, 45, 60, np.inf],
        labels=['Baby', 'Child', 'Teen', 'YoungAdult', 'Adult', 'Mature', 'Senior']
    ).astype(str)

    out['FareBand'] = pd.qcut(out['Fare'], q=5, duplicates='drop').astype(str)

    return out


TARGET = 'Survived'
y = train[TARGET].astype(int)

train_features = engineer_features(train.drop(columns=[TARGET]), train.drop(columns=[TARGET]))
test_features = engineer_features(test.copy(), train.drop(columns=[TARGET]))

drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Surname']
train_features = train_features.drop(columns=[c for c in drop_cols if c in train_features.columns])
test_features = test_features.drop(columns=[c for c in drop_cols if c in test_features.columns])

categorical = train_features.select_dtypes(include=['object', 'category']).columns.tolist()

for frame in [train_features, test_features]:
    for col in categorical:
        frame[col] = frame[col].astype(str).fillna('Missing')

train_features[categorical] = train_features[categorical].fillna('Missing')
test_features[categorical] = test_features[categorical].fillna('Missing')

print('Engineered train shape:', train_features.shape)
print('Engineered test shape:', test_features.shape)
print('Categorical features:', len(categorical))

## Winning Experiment 22 models

CAT_A and CAT_B were the strongest combination in Experiment 22. They are retrained independently on all available training rows and their probabilities are averaged equally.

In [ ]:
cat_a = CatBoostClassifier(
    iterations=1200,
    depth=5,
    learning_rate=0.025,
    loss_function='Logloss',
    l2_leaf_reg=6,
    random_seed=42,
    verbose=False
)

cat_b = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.025,
    loss_function='Logloss',
    l2_leaf_reg=8,
    random_seed=49,
    verbose=False
)

cat_a.fit(train_features, y, cat_features=categorical)
cat_b.fit(train_features, y, cat_features=categorical)

pred_a = cat_a.predict_proba(test_features)[:, 1]
pred_b = cat_b.predict_proba(test_features)[:, 1]

final_probability = (pred_a + pred_b) / 2
final_prediction = (final_probability >= 0.5).astype(int)

print('Predictions generated:', len(final_prediction))
print('Survived:', int(final_prediction.sum()))
print('Not survived:', int((1 - final_prediction).sum()))

In [ ]:
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': final_prediction
})

assert submission.shape == (418, 2)
assert list(submission.columns) == ['PassengerId', 'Survived']
assert submission['PassengerId'].equals(test['PassengerId'])
assert submission['PassengerId'].is_unique
assert set(submission['Survived'].unique()).issubset({0, 1})

output_path = SUB_DIR / 'submission_07.csv'
submission.to_csv(output_path, index=False)

print('Saved:', output_path)
print('Shape:', submission.shape)
display(submission.head(10))

## Submission description

Heavy CatBoost ensemble submission based on Experiment 22, where the equal-weight CAT_A + CAT_B blend achieved 0.8406 OOF accuracy. Both models use the engineered Titanic features and are retrained on the full training dataset before generating the 418 test predictions.